# Splitting a book into parts and chapters

The chapter-detection cascade: bookmarks first, heading lines second; then quote style and character names.

See the [README](README.md) for the why and the how.

In [ ]:
# Run from anywhere inside the repository: find its root so `marginalia` and the test helpers import,
# and read .env (the same settings the services use).
import os, sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / 'pyproject.toml').exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print(ROOT)

In [ ]:
from tests.pdfs import make_book, chapter, part
from marginalia.ingest.book import load_book
with_outline = make_book([part('PART ONE', outline='Part One'),
                          chapter('CHAPTER I', 'the river', outline='Chapter I', level=1),
                          chapter('CHAPTER II', 'the axe', outline='Chapter II', level=1),
                          part('PART TWO', outline='Part Two'),
                          chapter('CHAPTER I', 'the letter', outline='Chapter I', level=1)], title='Two Parts')
book = load_book(with_outline)
print('found by:', book.method, '| parts:', book.parts)
for c in book.chapters:
    print(f'{c.index:2} part {c.part} chapter {c.chapter}: {c.label!r} {c.title!r} ({len(c.text)} chars)')

## No bookmarks: heading lines
Without an outline, lines that look like headings ("CHAPTER ONE", "Chapter 4: Title", "Part II", "Prologue") are found on the pages.

In [ ]:
no_outline = make_book([chapter('CHAPTER ONE', 'the river', title='THE BRIDGE'),
                        chapter('CHAPTER TWO', 'the axe'), chapter('CHAPTER THREE', 'the letter')], title='Headings')
b2 = load_book(no_outline)
print(b2.method, [(c.label, c.title) for c in b2.chapters])

In [ ]:
from marginalia.ingest.structure import classify
for line in ['CHAPTER IV', 'Chapter 12: The Mirror', 'PART TWO', 'Prologue', 'Epilogue', 'Book One',
             'The rain fell on the old town.', 'Contents']:
    print(f'{line:32} -> {classify(line)}')

## A book without chapters is refused

In [ ]:
from marginalia.ingest.structure import NoChapters
prose = make_book([{'heading': 'Notes', 'body': ['Plain prose without any chapter heading. ' * 40] * 8}], title='Notes')
try:
    load_book(prose)
except NoChapters as e:
    print(e.code, '-', e)

## Quote style and names
Dialogue becomes its own searchable records later; the style decides which quotation marks open and close a line of dialogue.

In [ ]:
from marginalia.ingest.quotes import detect_quote_style
from marginalia.ingest.book import frequent_names
sample = '“Where are you going?” asked Sonia. ‘Out,’ said Raskolnikov. "Now," he added.'
print('quote style:', detect_quote_style(sample * 5))
print('names in the demo book:', book.names)
print(frequent_names(('Sonia met Raskolnikov. Dounia wrote to Raskolnikov. ' * 20), min_count=5))